In [15]:
import pandas as pd
import numpy as np

In [16]:
data = pd.read_csv(r"DataSet/DateFruit_Dataset.csv")
df = pd.DataFrame(data)
df.sample(5)

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
237,142413,1438.2170,527.6629,346.8893,0.7535,425.8238,0.9794,145415,0.7121,1.5211,...,2.4505,2.3521,2.4319,-14408605696,-12457070592,-11494690816,49.6982,46.8458,44.8598,DOKOL
214,138262,1403.3560,542.0650,326.1642,0.7987,419.5720,0.9935,139162,0.7554,1.6619,...,3.9672,2.9005,2.4017,-28823592960,-28359831552,-20046186496,70.3907,70.4653,59.7454,DOKOL
689,333356,2242.6880,880.4042,483.8072,0.8355,651.4922,0.9834,338967,0.6480,1.8197,...,15.1736,11.5855,5.3592,-7259139584,-9597211648,-12309225472,20.5187,25.1953,31.2370,SAFAVI
730,298760,2184.9109,842.4735,453.8198,0.8425,616.7601,0.9871,302671,0.8112,1.8564,...,11.8010,10.4741,4.0698,-7861085184,-10529727488,-17813413888,26.3517,30.3071,39.6009,SAFAVI
888,242092,1875.0250,700.3940,441.3842,0.7764,555.1946,0.9930,243806,0.7163,1.5868,...,2.1165,2.3162,2.4141,-19969982464,-17911033856,-13709812736,41.1247,38.6405,36.6200,SOGAY


In [17]:
X = df.drop(["Class"],axis=1)
y =df["Class"]

df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [18]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
scaler = StandardScaler()
LE = LabelEncoder()
y=LE.fit_transform(y)


In [30]:
from sklearn.model_selection import train_test_split

X_train ,X_test,Y_train ,Y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## convert data into tensor

In [32]:
x_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(Y_train,dtype=torch.long)

x_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(Y_test, dtype=torch.long )

### Conver into tensor dataset and dataloder

In [34]:
train_dataset = TensorDataset(x_train_tensor,y_train_tensor)
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)


train_loder = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loder = DataLoader(train_dataset,batch_size=32)

### Build ANN MOdel

In [35]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    
    def forward(self,x):
        return self.model(x)

### creaet the object define optimizer criteria

In [37]:
model = ANN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### training the nural Network

In [42]:
epoch = 100
for  epoch in range(epoch):
    model.train()
    running_loss=0.0

    for xb,yb in train_loder:
        optimizer.zero_grad()
        output = model(xb)

        loss = criteria(output,yb)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
    
    traing_loss = running_loss / len(train_loder)
    print(f"epoch = {epoch+1} / {epoch}==> loss -: {traing_loss}")

epoch = 1 / 0==> loss -: 1.2763094804212966e-07
epoch = 2 / 1==> loss -: 1.264971769747617e-07
epoch = 3 / 2==> loss -: 1.2520143941886352e-07
epoch = 4 / 3==> loss -: 1.2439160462060867e-07
epoch = 5 / 4==> loss -: 1.2309586448706225e-07
epoch = 6 / 5==> loss -: 1.2260996647924263e-07
epoch = 7 / 6==> loss -: 1.2864910042578845e-07
epoch = 8 / 7==> loss -: 1.2001848962005174e-07
epoch = 9 / 8==> loss -: 1.4230066603321026e-07
epoch = 10 / 9==> loss -: 1.1844509451502708e-07
epoch = 11 / 10==> loss -: 1.1694111139340983e-07
epoch = 12 / 11==> loss -: 1.1596930778963756e-07
epoch = 13 / 12==> loss -: 1.14511599642211e-07
epoch = 14 / 13==> loss -: 1.1707997649640126e-07
epoch = 15 / 14==> loss -: 1.128919261744019e-07
epoch = 16 / 15==> loss -: 1.1143422138660675e-07
epoch = 17 / 16==> loss -: 1.1078635339546787e-07
epoch = 18 / 17==> loss -: 1.0965257846645463e-07
epoch = 19 / 18==> loss -: 1.0884274313722356e-07
epoch = 20 / 19==> loss -: 1.0759328184916722e-07
epoch = 21 / 20==> loss

### Evaluate 

In [43]:
total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loder:
        output = model(xb)
        _, predicted = torch.max(output,1)
        correct += (predicted == yb).sum().item()

        total += yb.size(0)

print("accracy = ",correct/total * 100)

accracy =  100.0
